# YOLOv8 Card Detector Training

This notebook trains a **YOLOv8-nano** model to detect playing cards in images (Stage 1 of the 24 Game Solver pipeline).

- **Input:** raw photos containing playing cards
- **Output:** bounding boxes around each card (single class: `card`)
- **Export:** TFLite FP16 at 320×320 for on-device inference in Flutter

> **Runtime:** Set to GPU (Runtime → Change runtime type → T4 GPU) before running.

## Setup

Install dependencies and import libraries.

In [ ]:
# Install dependencies
!pip install -q ultralytics kaggle

import os
import glob
from pathlib import Path
from ultralytics import YOLO

## Data

Download the [Playing Cards Object Detection Dataset](https://www.kaggle.com/datasets/andy8744/playing-cards-object-detection-dataset) from Kaggle.

This dataset contains ~7,000 images of playing cards with YOLO-format bounding box annotations (52 classes). We'll remap all classes to a single `card` class for Stage 1 detection.

**Before running:** Set your Kaggle API token as an environment variable:
```python
import os
os.environ["KAGGLE_API_TOKEN"] = "your_token_here"
```

In [ ]:
# Download playing cards object detection dataset from Kaggle
!kaggle datasets download -d andy8744/playing-cards-object-detection-dataset -p data/ --unzip

# The dataset extracts with train/test/valid splits already in YOLO format
DATA_DIR = "data"

# Show dataset structure
for split in ["train", "test", "valid"]:
    img_dir = os.path.join(DATA_DIR, split, "images")
    if os.path.isdir(img_dir):
        print(f"{split}: {len(os.listdir(img_dir))} images")
    else:
        print(f"{split}: directory not found at {img_dir}")

### Remap Classes

The dataset may have fine-grained labels (e.g. `ace_of_spades`, `2_of_hearts`, …). We collapse all 52 card classes into a single class `card` (class ID 0) since Stage 1 only needs to *locate* cards — Stage 2 will classify the rank/suit.

In [ ]:
# Remap all class labels to single class 0 ("card")
# YOLO format: class_id center_x center_y width height (normalized)

def remap_labels_to_single_class(labels_dir: str) -> int:
    """Rewrite all YOLO label files to use class 0 for every object."""
    count = 0
    for label_file in glob.glob(os.path.join(labels_dir, "*.txt")):
        with open(label_file, "r") as f:
            lines = f.readlines()

        remapped = []
        for line in lines:
            parts = line.strip().split()
            if len(parts) >= 5:
                # Replace class ID (first field) with 0
                parts[0] = "0"
                remapped.append(" ".join(parts))

        with open(label_file, "w") as f:
            f.write("\n".join(remapped) + "\n" if remapped else "")
        count += 1
    return count

for split in ["train", "test", "valid"]:
    labels_dir = os.path.join(DATA_DIR, split, "labels")
    if os.path.isdir(labels_dir):
        n = remap_labels_to_single_class(labels_dir)
        print(f"Remapped {n} label files in {split}/")
    else:
        print(f"No labels dir found for {split}")

### Create data.yaml

Write the YOLO dataset config file that points to the train/val splits and defines the single `card` class.

In [ ]:
# Create YOLO data config pointing to the Kaggle dataset
data_yaml = f"""
path: /content/data
train: train/images
val: valid/images
test: test/images

nc: 1
names:
  0: card
"""

with open(os.path.join(DATA_DIR, "data.yaml"), "w") as f:
    f.write(data_yaml.strip())

print("Created data.yaml")
print(open(os.path.join(DATA_DIR, "data.yaml")).read())

## Training

Train **YOLOv8-nano** for up to 100 epochs with early stopping (patience=15). Training at 640×640 gives better feature quality before we export at 320×320 for inference.

Expected training time on a T4 GPU: ~20–40 minutes depending on dataset size.

In [ ]:
# Train YOLOv8-nano
model = YOLO("yolov8n.pt")

results = model.train(
    data="data/data.yaml",
    epochs=100,
    imgsz=640,       # train at 640 for better features
    batch=16,
    patience=15,     # early stopping
    project="outputs",
    name="yolo_card_detector",
    exist_ok=True,
)

print("Training complete!")
print(f"Best model: outputs/yolo_card_detector/weights/best.pt")

## Evaluation

Evaluate the best checkpoint on the validation set. We target **mAP@0.5 > 0.90** for reliable card detection in the Flutter app.

In [ ]:
# Evaluate on validation set
best_model = YOLO("outputs/yolo_card_detector/weights/best.pt")
metrics = best_model.val(data="data/data.yaml")

print(f"mAP@0.5:     {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision:    {metrics.box.mp:.4f}")
print(f"Recall:       {metrics.box.mr:.4f}")

# Target: mAP@0.5 > 0.90
assert metrics.box.map50 > 0.85, f"mAP@0.5 too low: {metrics.box.map50:.4f}"

### Visualize Predictions

Run inference on a sample of validation images and display the annotated results.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

# Run predictions on sample validation images
val_images = glob.glob("data/valid/images/*.jpg")[:6]
if not val_images:
    val_images = glob.glob("data/valid/images/*.png")[:6]

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
for ax, img_path in zip(axes.flat, val_images):
    results = best_model.predict(img_path, conf=0.5, verbose=False)
    annotated = results[0].plot()
    ax.imshow(annotated[:, :, ::-1])  # BGR to RGB
    ax.axis("off")
    ax.set_title(Path(img_path).name)

plt.suptitle("YOLOv8-nano Card Detector — Validation Predictions", fontsize=16)
plt.tight_layout()
os.makedirs("outputs/yolo_card_detector", exist_ok=True)
plt.savefig("outputs/yolo_card_detector/val_predictions.png", dpi=150)
plt.show()

## Export

Export the trained model to **TFLite FP16** at 320×320 resolution for on-device inference.

After exporting, copy the `.tflite` file to `flutter_app/assets/models/card_detector.tflite`.

In [ ]:
# Export to TFLite at inference size 320x320
# FP16 quantization for good accuracy + small size
exported_path = best_model.export(
    format="tflite",
    imgsz=320,
    int8=False,
    half=True,  # FP16
)

import os
tflite_files = glob.glob("outputs/yolo_card_detector/weights/*float16.tflite") + \
               glob.glob("outputs/yolo_card_detector/weights/*.tflite")
for f in tflite_files:
    size_mb = os.path.getsize(f) / (1024 * 1024)
    print(f"{f}: {size_mb:.1f} MB")

print("\nCopy the .tflite file to flutter_app/assets/models/card_detector.tflite")